In [ ]:
import imageio.v3 as iio
import matplotlib.pyplot as plt
from pathlib import Path 
import glob, os
import numpy as np
from motionnet.dataset.van_hateren_utils import load_vanhateren_raw, log_image, random_crop
from motionnet.dataset import GetNaturalMovies, animate_sample, plot_examples
from scipy.ndimage import uniform_filter, spline_filter, map_coordinates
from tqdm import tqdm
import seaborn as sns
sns.set_context("talk")

In [ ]:
## GLOBAL VARS
N_IMAGES = 30
RNG = np.random.default_rng(1)
DATA_PATH = Path("/home/guardomayas/NUIN/van_hateren/vanhateren_iml")
DPP = 1/60 # Van Hateren images are 1/60 deg per pixel 

# 1. Self contained notebook to illustrate how to make Van Hateren Dataset fly like

In [ ]:
iml_files = np.array(sorted(DATA_PATH.glob("*.iml")), dtype=object)
selected_files = iml_files[RNG.choice(len(iml_files), size=N_IMAGES, replace=False)]

images = np.empty((N_IMAGES, 1024, 1536), dtype=np.float32)

for i, f in enumerate(tqdm(selected_files, desc="Loading images")):
    img = log_image(f)
    images[i] = img    

In [ ]:
fig, axes = plot_examples(images, 5, 5, random_seed=10, mode="log")

In [ ]:
H,W = images.shape[1:3]

print(f"Image dimensions: {H}x{W}")
x_min = -0.5 * W * DPP
x_max =  0.5 * W * DPP
y_min = -0.5 * H * DPP
y_max =  0.5 * H * DPP

print(f"x range: {x_min:.2f} to {x_max:.2f} deg")
print(f"y range: {y_min:.2f} to {y_max:.2f} deg")

fig, axes = plt.subplots(1, 1, figsize=(8, 6))
img = axes.imshow(images[26], cmap="gray", extent=[x_min, x_max, y_min, y_max])
axes.set_xlabel("Azimuth (deg)")
axes.set_ylabel("Elevation (deg)")


## Fly sampling

Similarly to `flyvis`, we will assert that the fly's ommatidia is 13pixels using a box mean or gaussianfilter.
    
*Note:*  this is the same number we get by using Meyer's dataset, so I believe that's how they compute it.

In [ ]:
from scipy.ndimage import gaussian_filter

KERNEL_SIZE = 13
EYE_SIZE    = 28 # Number of receptors   
THETA       = 5.3 #Half the angle of view of the eye in degrees
DPP_FLY     = THETA/KERNEL_SIZE #Our resampling dpp
DELTA_RHO_PX = KERNEL_SIZE * 1.08          # Δρ/Δφ ≈ 1.08 → 14.04 px FWHM
SIGMA        = DELTA_RHO_PX / 2.3548   # ≈ 5.96


img         = images[26]
# blur        = uniform_filter(img, size=KERNEL_SIZE)
blur        = gaussian_filter(img, sigma=SIGMA, mode="reflect", truncate=3.0)
coeffs      = spline_filter(blur, order=3, output=np.float32) #spline filter the image to get the coefficients for interpolation, avoids aliasing artifacts

#receptor grid, cartesian for now (source px), centered on 0 
off         = (np.arange(EYE_SIZE) - (EYE_SIZE-1)/2) * KERNEL_SIZE 
gy, gx      = np.meshgrid(off, off, indexing="ij")
cx, cy      = W/2, H/2  #center of the eye in degrees

frame = map_coordinates(coeffs, [gy + cy, gx + cx], order=3,
                        mode="reflect", prefilter=False) #interpolate the image at the receptor locations

# --- crop bounds, per axis --------------------------------------
half_px  = (EYE_SIZE - 1) / 2 * KERNEL_SIZE                      
y0, y1   = int(np.floor(cy - half_px)), int(np.ceil(cy + half_px)) + 1
x0, x1   = int(np.floor(cx - half_px)), int(np.ceil(cx + half_px)) + 1

crop     = blur[y0:y1, x0:x1]

fig, axs = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
axs[0].imshow(blur, cmap="gray")
axs[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                               ec="r", fc="none", lw=1.5))
axs[0].set_title("full 1024x1536")

axs[1].imshow(crop, cmap="gray")
axs[1].set_title(f"{crop.shape} source region")

half = (EYE_SIZE - 1) / 2 * THETA + THETA / 2                   

axs[2].imshow(frame, cmap="gray", interpolation="nearest",
              extent=[-half, half, -half, half])
axs[2].set_xlabel("Azimuth (deg)")
axs[2].set_ylabel("Elevation (deg)")
axs[2].set_title(f"{EYE_SIZE}x{EYE_SIZE} receptors")
axs[2].set_xticks([-75, -50, -25, 0, 25, 50, 75])
axs[2].set_yticks([-75, -50, -25, 0, 25, 50, 75])



## Build naturilistic movie

In [ ]:
noise_std = 0.0
d = GetNaturalMovies(DATA_PATH, n_images=N_IMAGES, start_jitter=100, samples_per_image=2, noise_std=noise_std,
                     frames_per_segment=75*2, vel_std_deg_s=(120, 70))
# for i in range(100):
#     d[i]
# print(f"acceptance {100 * d.acceptance_rate:.0f}%")

In [ ]:
def contrast_movies(d, idx=0, gains=(1.0, 0.5, 0.25), noise_std=0.1,
                    frames=(20, 50, 60), seed=0, fixed_scale=2.5):
    """Same scene and velocity trace at several contrasts.

    Reproduces the dataset's order of operations on an already-rendered
    sample: gain on the zero-mean movie, then fixed-variance noise, which
    is why the noise is masked out of the gray padding here (the dataset
    adds noise before padding, so its padding is clean).

    `d` must have contrast_range=None and noise_std=0, or the gain and
    noise applied here stack on top of the dataset's own.

    fixed_scale: imshow vmin/vmax. Set None to autoscale each panel --
    autoscaling shows what survives the noise, a fixed scale shows how
    much signal is actually left.
    """
    if d.noise_std > 0 or d.contrast_range is not None:
        raise ValueError("pass a dataset with noise_std=0 and "
                         "contrast_range=None; this function applies both")

    s = d[idx]
    clean = s["movie"].numpy()[:, 0]                 # (T, H, W)
    vel   = s["vel_deg_s"].numpy()
    t_s   = np.arange(len(vel)) / d.fps
    gray  = d.gray_frames
    rms0  = clean[gray:].std()                       # scene only, not padding

    rng = np.random.default_rng(seed)
    noise = np.zeros_like(clean)                     # padding stays clean
    noise[gray:] = rng.normal(0, noise_std, clean[gray:].shape)

    n_g, n_f = len(gains), len(frames)
    kw = {} if fixed_scale is None else dict(vmin=-fixed_scale,
                                             vmax=fixed_scale)

    fig, axes = plt.subplots(n_g, n_f + 1, squeeze=False, layout="constrained",
                             figsize=(3.2 * (n_f + 1), 3.0 * n_g))

    for r, g in enumerate(gains):
        m = clean * g + noise
        for c, f in enumerate(frames):
            axes[r, c].imshow(m[f], cmap="gray", interpolation="nearest", **kw)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(f"t = {f / d.fps:.2f} s")
        axes[r, 0].set_ylabel(f"gain {g}\nRMS {g * rms0:.2f}\n"
                              f"SNR {g * rms0 / noise_std:.1f}", fontsize=9)

        ax = axes[r, n_f]
        ax.plot(t_s, vel[:, 0], label="vx", lw=1)
        ax.plot(t_s, vel[:, 1], label="vy", lw=1)
        for f in frames:
            ax.axvline(f / d.fps, color="k", lw=0.8, alpha=0.4)
        ax.axhline(0, ls="--", c="k", lw=0.5)
        ax.set(ylim=(-250, 250), ylabel="velocity (deg/s)")
        if r == n_g - 1:
            ax.set_xlabel("time (s)")
    axes[0, n_f].legend(fontsize=8)

    scale = "fixed scale" if fixed_scale else "autoscaled"
    fig.suptitle(f"contrast sweep — {scale}, noise_std = {noise_std}")
    return fig
contrast_movies(d, idx=0, gains = [1.0], noise_std=0.05, fixed_scale=None)
# contrast_movies(d, idx=0, noise_std=0.05, fixed_scale=None)

In [ ]:
rms = np.array([d[i]["contrast_rms"].item() for i in range(len(d))])
print(f"RMS across samples: {rms.mean():.2f} ± {rms.std():.2f}, "
      f"range {rms.min():.2f}–{rms.max():.2f}")

In [ ]:
from IPython.display import HTML, display
for idx in (0, 5):
    anim = animate_sample(d, idx=idx, stride=3)
    display(HTML(anim.to_jshtml()))
    anim.save(f"sample_{idx}_noise_{noise_std}.gif", writer="pillow", fps=d.fps / 3, dpi=110)
    plt.close(anim._fig)          # suppress the static figure